In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout
from tensorflow.keras.optimizers import SGD, RMSprop, Adam, AdamW
from tensorflow.keras import regularizers, models, layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import optuna
import wandb
import gc
import cv2

import os
import sqlite3

2026-03-20 08:13:43.353960: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-20 08:13:43.354002: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-20 08:13:43.354843: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-20 08:13:43.360024: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
tf.keras.backend.clear_session()

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print("TensorFlow is using the GPU \n", gpus)
else:
    print("No GPU detected.")

TensorFlow is using the GPU 
 [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2026-03-20 08:13:50.069983: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-20 08:13:50.079917: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-20 08:13:50.085544: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [3]:
import wandb 

from wandb.integration.keras import WandbMetricsLogger

wandb.require("core")
wandb.login()

wandb: WARNING `wandb.require('core')` is a no-op as it is now the default behavior.
wandb: Currently logged in as: emmdaz (emmdaz-zzz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import uproot
from variables_calculator import variables_calculator

file = "/tf/Higgs-Boson-LHC-Collision-Detector/sigfcc_350.root"

data = uproot.open(file)
tree = data["Delphes"]

higgs_b = variables_calculator.inv_m(file, "jet_1", "jet_2", flavor1 = 3, flavor2 = 5)
higgs_b = higgs_b[["pt_1", "pt_2", "eta_1", "eta_2", "phi_1", "phi_2", "event_1", "inv_m"]].copy()
higgs_b.columns = ["PRI_jet_leading_pt", "PRI_jet_subleading_pt", "PRI_jet_leading_eta", "PRI_jet_subleading_eta", "PRI_jet_leading_phi",
                  "PRI_jet_subleading_phi", "event_1", "DER_mass_jet_jet"]

PRI_jet_num = variables_calculator.PRI_jet_num(file)
PRI_jet_all = variables_calculator.PRI_jet_all_pt(file)
met = variables_calculator.met(file)

jets_data = pd.merge(PRI_jet_all, PRI_jet_num, on = "event_1")
jets_data = pd.merge(jets_data, met, on = "event_1")
jets_data = pd.merge(jets_data, higgs_b, on = "event_1")

jets_data["label"] = np.full(len(jets_data), 1)

muon_data = variables_calculator.inv_m(file, "muon", "muon")
electron_data = variables_calculator.inv_m(file, "electron", "electron")
lepton_data = pd.concat([muon_data, electron_data])

lepton_data = lepton_data[["event_1", "inv_m"]].copy()
lepton_data.columns = ["event_1", "DER_mass_lep"]

signal = pd.merge(jets_data, lepton_data, on = "event_1")
signal = signal.drop(columns = ["event_1"])

file = "/tf/Higgs-Boson-LHC-Collision-Detector/bgd240zh.root"

data = uproot.open(file)
tree = data["Delphes"]

higgs_b = variables_calculator.inv_m(file, "jet_1", "jet_2", flavor1 = 3, flavor2 = 5)
higgs_b = higgs_b[["pt_1", "pt_2", "eta_1", "eta_2", "phi_1", "phi_2", "event_1", "inv_m"]].copy()
higgs_b.columns = ["PRI_jet_leading_pt", "PRI_jet_subleading_pt", "PRI_jet_leading_eta", "PRI_jet_subleading_eta", "PRI_jet_leading_phi",
                  "PRI_jet_subleading_phi", "event_1", "DER_mass_jet_jet"]

PRI_jet_num = variables_calculator.PRI_jet_num(file)
PRI_jet_all = variables_calculator.PRI_jet_all_pt(file)
met = variables_calculator.met(file)

jets_data = pd.merge(PRI_jet_all, PRI_jet_num, on = "event_1")
jets_data = pd.merge(jets_data, met, on = "event_1")
jets_data = pd.merge(jets_data, higgs_b, on = "event_1")

jets_data["label"] = np.full(len(jets_data), 0)

muon_data = variables_calculator.inv_m(file, "muon", "muon")
electron_data = variables_calculator.inv_m(file, "electron", "electron")
lepton_data = pd.concat([muon_data, electron_data])

lepton_data = lepton_data[["event_1", "inv_m"]].copy()
lepton_data.columns = ["event_1", "DER_mass_lep"]

noise1 = pd.merge(jets_data, lepton_data, on = "event_1")
noise1 = noise1.drop(columns = ["event_1"])

# Noise 2
file = "/tf/Higgs-Boson-LHC-Collision-Detector/bgd365eez.root"

data = uproot.open(file)
tree = data["Delphes"]

higgs_b = variables_calculator.inv_m(file, "jet_1", "jet_2", flavor1 = 3, flavor2 = 5)
higgs_b = higgs_b[["pt_1", "pt_2", "eta_1", "eta_2", "phi_1", "phi_2", "event_1", "inv_m"]].copy()
higgs_b.columns = ["PRI_jet_leading_pt", "PRI_jet_subleading_pt", "PRI_jet_leading_eta", "PRI_jet_subleading_eta", "PRI_jet_leading_phi",
                  "PRI_jet_subleading_phi", "event_1", "DER_mass_jet_jet"]

PRI_jet_num = variables_calculator.PRI_jet_num(file)
PRI_jet_all = variables_calculator.PRI_jet_all_pt(file)
met = variables_calculator.met(file)

jets_data = pd.merge(PRI_jet_all, PRI_jet_num, on = "event_1")
jets_data = pd.merge(jets_data, met, on = "event_1")
jets_data = pd.merge(jets_data, higgs_b, on = "event_1")

jets_data["label"] = np.full(len(jets_data), 0)

muon_data = variables_calculator.inv_m(file, "muon", "muon")
electron_data = variables_calculator.inv_m(file, "electron", "electron")
lepton_data = pd.concat([muon_data, electron_data])

lepton_data = lepton_data[["event_1", "inv_m"]].copy()
lepton_data.columns = ["event_1", "DER_mass_lep"]

noise2 = pd.merge(jets_data, lepton_data, on = "event_1")
noise2 = noise2.drop(columns = ["event_1"])

noise = pd.concat([noise1, noise2])

bg_sample1 = noise.sample(frac = 0.5, random_state = 4)
bg_sample2 = noise.sample(frac = 0.3, random_state = 4)
bg_sample3 = noise.sample(frac = 0.2, random_state = 4)

sg_sample1 = signal.sample(n = len(bg_sample1)*3, random_state = 4)
sg_sample2 = signal.sample(n = len(bg_sample2)*3, random_state = 4)
sg_sample3 = signal.sample(n = len(bg_sample3)*3, random_state = 4)

train = pd.concat([sg_sample1, bg_sample1])
test = pd.concat([sg_sample2, bg_sample2])
val = pd.concat([sg_sample3, bg_sample3])

train = train.sample(frac = 1, random_state = 5).reset_index(drop = True)
test = test.sample(frac = 1, random_state = 5).reset_index(drop = True)
val = val.sample(frac = 1, random_state = 5).reset_index(drop = True)

# Now we can create the subsets:
X_train, X_test, X_val = train.drop(columns = ["label"]), test.drop(columns = ["label"]), val.drop(columns = ["label"])
y_train, y_test, y_val = train["label"], test["label"], val["label"]

# And we stardardize the data:

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Check the sizes
print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

Train size: 1700
Validation size: 680
Test size: 1020


/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [5]:
print(len(sg_sample1))
print(len(bg_sample1))

1275
425


In [6]:
signal.head()

,PRI_jet_all_pt,PRI_jet_num,PRI_met,PRI_met_phi,PRI_jet_leading_pt,PRI_jet_subleading_pt,PRI_jet_leading_eta,PRI_jet_subleading_eta,PRI_jet_leading_phi,PRI_jet_subleading_phi,DER_mass_jet_jet,label,DER_mass_lep
0,170.760605,3,4.302101,-1.388696,64.256523,36.597229,-0.113773,-1.709504,2.964018,-1.596112,113.084663,1,85.018837
1,304.227875,3,4.129518,2.948901,39.575764,127.212677,-0.525154,0.348251,1.704656,-3.009968,118.901390,1,82.369003
2,283.492126,3,2.992920,1.142674,137.516327,11.906924,0.033779,-1.482367,-1.903679,-0.247148,89.987083,1,92.270065
3,242.444855,3,5.493720,-2.066545,41.898006,102.129112,0.853439,0.485302,2.301678,-1.990217,112.418762,1,83.204475
4,230.607391,3,2.175927,1.302745,63.231850,81.647446,1.091058,0.237706,-0.946260,0.968609,133.423233,1,83.884644


In [7]:
train.head()

,PRI_jet_all_pt,PRI_jet_num,PRI_met,PRI_met_phi,PRI_jet_leading_pt,PRI_jet_subleading_pt,PRI_jet_leading_eta,PRI_jet_subleading_eta,PRI_jet_leading_phi,PRI_jet_subleading_phi,DER_mass_jet_jet,label,DER_mass_lep
0,239.087677,3,2.918416,-1.136685,97.319443,39.325672,-0.874773,0.070722,1.735062,0.158167,106.700081,1,89.485268
1,143.534821,3,11.685090,0.848280,64.629486,65.953690,0.791193,-0.148890,-2.195308,0.958930,145.266388,0,89.182938
2,198.904312,3,21.869644,-1.976044,28.644110,87.506470,0.829670,0.687386,1.511135,-1.952377,99.093338,1,92.251511
3,261.334290,4,7.230958,-0.462780,100.688171,16.901278,-0.821157,1.183019,-0.490292,-0.689705,97.571991,1,92.205437
4,284.861328,3,6.746761,-0.707597,119.034531,44.812328,0.146229,0.849623,3.042453,-1.610380,118.546295,1,93.519783


In [8]:
# We define a function to create residual blocks (considering a layer regularizer L2)

def residual_block(x, n, activation, dropout, dropout_rate, regularizer, r_2):
        
    residual = x  
        
    if dropout == "y":
        # Main path
        x = layers.Dense(n, activation = activation, kernel_regularizer = regularizers.l2(r_2))(x)
        x = layers.BatchNormalization()(x)
        
        # Intermediate dropout layer
        x = layers.Dropout(dropout_rate)(x)
            
        # Linear layer
        x = layers.Dense(n, kernel_regularizer = regularizers.l2(r_2))(x)
        x = layers.BatchNormalization()(x)
            
    else: 
        # Main path
        x = layers.Dense(n, activation = activation, kernel_regularizer = regularizers.l2(r_2))(x)
        x = layers.BatchNormalization()(x)
            
        # Linear layer
        x = layers.Dense(n, kernel_regularizer = regularizers.l2(r_2))(x)
        x = layers.BatchNormalization()(x)

    # Project shortcut to same dimension
    residual = layers.Dense(n)(residual)

    # Conection residual sum
    x = layers.add([x, residual]) 
    x = layers.Activation(activation)(x)
        
    return x

In [9]:
# We create the function to start the study and optimizarion using Optuna
def objective(trial):

    tf.keras.backend.clear_session()

    inputs = layers.Input(shape = (X_train.shape[1],))
    
    #############################################################################################################
    
    # Optuna suggests activation function for all layers
    activation = trial.suggest_categorical("activation", ["relu", "relu6", "leaky_relu"])
    
    # Optuna suggests regularizer L2 value
    regularizer = "L2"
    r_2 = trial.suggest_float("regularizer_value_2", 1e-7, 1e-5, log = True)
    
    # Optuna suggest the number of layers
    n_layers = trial.suggest_int("N_layers", 15,20)
    
    # Optuna suggests learning rate value and an optimizer
    lr = trial.suggest_float("learning_rate", 2.5e-4, 1e-3, log = True)
    
    optimizer_name = trial.suggest_categorical("optimizer", ["sgd", "adam", "rmsprop", "adamw"])
                              
    if optimizer_name == "sgd":
        optimizer = tf.keras.optimizers.SGD(learning_rate = lr)
    elif optimizer_name == "adam":
        optimizer = tf.keras.optimizers.Adam(learning_rate = lr)
    elif optimizer_name == "rmsprop":
        optimizer = tf.keras.optimizers.RMSprop(learning_rate = lr)
    elif optimizer_name == "adamw":
        optimizer = tf.keras.optimizers.AdamW(learning_rate = lr)              
    
    #############################################################################################################
    
    # First layer

    # Optuna suggest number of neurons for the first layer

    N = trial.suggest_int("N_1st_layer", 128, 256)
    
    x = layers.Dense(N, input_shape = (X_train.shape[1],))(inputs)
    x = layers.Activation(activation)(x)
    x = layers.BatchNormalization()(x)
    
    # Optuna suggests neurons for the residual blocks and if using Dropout block
    
    dropout_per_layer = []
    dropout_percentage_per_layer = []
            
    dropping_out = trial.suggest_categorical("Dropout", ["y", "n"])

    N_per_layer = []
    
    for i in range(n_layers):
        
        n = trial.suggest_int(f"N_{i+1}_layer", 128, 256)
        N_per_layer.append(n)
                              
        dropout_rate = trial.suggest_float(f"Dropout_value_L{i+2}",0.1, 0.15)
        
        # i-th residual block:
        
        # Choosing between Dropout or a regulizer
        
        if dropping_out == "y":
            dropout_percentage_per_layer.append(dropout_rate)
            x = residual_block(x, n, activation, "y", dropout_rate, regularizer, r_2)

        else:
            dropout_percentage_per_layer.append(0.0)
            x = residual_block(x, n, activation, "n", dropout_rate, regularizer, r_2)            
            
    x = layers.Dropout(0.4)(x)  
    outputs = layers.Dense(1, activation = "sigmoid")(x)
    model = models.Model(inputs, outputs)
                              
    model.compile(optimizer = optimizer,
                  loss = "binary_crossentropy",
                  metrics = ["accuracy",
                             tf.keras.metrics.Precision(),
                             tf.keras.metrics.AUC(curve = "ROC"),
                             tf.keras.metrics.AUC(curve = "PR")])
    
    #############################################################################################################

    wandb.init(
        project = "Residual-SnB-Trials-HB-No-Balanced-1.0",
        name = f"Trial_{trial.number}",
        reinit = True,
        config = {
            "Units_1": N,
            "Units_per_layer": N_per_layer,
            "activation": activation,
            "n_layers": n_layers,
            "regularizer": regularizer,
            "r_value2": r_2,
            "Dropout": dropping_out, 
            "dropout_percentage_per_layer": dropout_percentage_per_layer,
            "learning_rate": lr,
            "optimizer": optimizer_name        }
    )
    
    #############################################################################################################
    
    """
    Callbacks
    """
    early_stopping = EarlyStopping(monitor = "val_accuracy", patience = 7, restore_best_weights = True)
    lr_reduction = ReduceLROnPlateau(monitor = "val_loss", factor = 0.1, patience = 5)
#     tensorboard_cb = TensorBoard(log_dir = "/workspace/Optuna-Trials/Plant-Pathology-Classificator/tf_debug", histogram_freq = 1, write_graph = True,
#                                  write_images = False)
    
    #############################################################################################################
    
    """
    Creación del modelo
    """
    
    try:
        print(model.summary())
    
        history = model.fit(
            X_train, y_train,
            validation_data = (X_test, y_test),
            batch_size = 32,
            epochs = 200,
            verbose = 1, 
            callbacks = [WandbMetricsLogger(log_freq = 5), early_stopping, lr_reduction]
        )

        val_loss = min(history.history["val_loss"])
        val_accuracy = max(history.history["val_accuracy"])
        
        train_loss = min(history.history["loss"])
        train_accuracy = max(history.history["accuracy"])
    
    except tf.errors.ResourceExhaustedError as e:
        
        print(f"Intento {trial.number} falló debido a: {e}")
        
        tf.keras.backend.clear_session()
        wandb.finish()
        gc.collect()
        
        return float("inf")

    except Exception as e:
        
        print(f"Intento {trial.number} falló. Unexpected error: {e}")
        
        tf.keras.backend.clear_session()
        wandb.finish()
        gc.collect()
        
        return float("inf")
    
    # score = val_loss + 0.1 * (train_loss - val_loss)
    
    score = val_accuracy
    
    # score = train_loss 
    
    tf.keras.backend.clear_session()
    gc.collect()
    wandb.finish()

    return 1-score

In [10]:
tf.keras.backend.clear_session()
gc.collect()

from tensorflow.keras import backend as K

K.clear_session()

In [ ]:
study = optuna.create_study(
    study_name = "Residual-Trials_HB(Balanced)-1.0",
    direction = "minimize",
    storage = "sqlite:////workspace/Optuna-Trials/ResNet_SnB(No-Balanced)_study.db",
    load_if_exists = True
)

study.optimize(objective, n_trials = 500, n_jobs = 1)

[I 2026-03-20 08:14:18,189] Using an existing study with name 'Residual-Trials_HB(Balanced)-1.0' instead of creating a new one.
2026-03-20 08:14:18.293251: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-20 08:14:18.296383: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-20 08:14:18.298636: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 12)]                 0         []                            
                                                                                                  
 dense (Dense)               (None, 180)                  2340      ['input_1[0][0]']             
                                                                                                  
 activation (Activation)     (None, 180)                  0         ['dense[0][0]']               
                                                                                                  
 batch_normalization (Batch  (None, 180)                  720       ['activation[0][0]']          
 Normalization)                                                                               

2026-03-20 08:14:34.401649: I external/local_xla/xla/service/service.cc:168] XLA service 0x7865b40fe7e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-20 08:14:34.401674: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2026-03-20 08:14:34.406199: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-20 08:14:34.421446: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8906
I0000 00:00:1773994474.477276  125670 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


54/54 [==============================] - 25s 56ms/step - loss: 0.5964 - accuracy: 0.7641 - precision: 0.8276 - auc: 0.7830 - auc_1: 0.9117 - val_loss: 0.3980 - val_accuracy: 0.8814 - val_precision: 0.9640 - val_auc: 0.9459 - val_auc_1: 0.9838 - lr: 9.8690e-04
Epoch 2/200
54/54 [==============================] - 2s 33ms/step - loss: 0.3877 - accuracy: 0.8600 - precision: 0.8998 - auc: 0.9123 - auc_1: 0.9673 - val_loss: 0.3845 - val_accuracy: 0.8539 - val_precision: 0.9196 - val_auc: 0.9263 - val_auc_1: 0.9775 - lr: 9.8690e-04
Epoch 3/200
54/54 [==============================] - 2s 34ms/step - loss: 0.3642 - accuracy: 0.8635 - precision: 0.8966 - auc: 0.9234 - auc_1: 0.9739 - val_loss: 0.2656 - val_accuracy: 0.8971 - val_precision: 0.9115 - val_auc: 0.9660 - val_auc_1: 0.9889 - lr: 9.8690e-04
Epoch 4/200
54/54 [==============================] - 2s 34ms/step - loss: 0.3794 - accuracy: 0.8706 - precision: 0.9055 - auc: 0.9174 - auc_1: 0.9642 - val_loss: 0.2514 - val_accuracy: 0.9382 - val_

batch/accuracy,▁▁▂▂▂▂▄▄▆▅▃▄▅▅▅▆▆▆▆▇▇▆▆▆▅▇▇▇█▇▇▇█▇██████
batch/auc,▁▂▆▆▆▆▆▆▇▇█▇▇▇██████████████████████████
batch/auc_1,▁▂▂▆▂▆▆▅▅▇███▇█▇▇▇▇█▇▇▇████▇▇███████████
batch/batch_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇█
batch/learning_rate,██████████████████████████████▁▁▁▁▁▁▁▁▁▁
batch/loss,█▄▃▄▂▂▂▃▂▂▂▂▂▁▁▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/precision,▁▃▄▅▄▆▂▇▇▇▇█▇▇▇▇██▇▇██▇▇███▇██▇█████████
epoch/accuracy,▁▄▄▄▆▆▇▆▆▇▇▇▇▇▇▇▇▇▇█▇█▇▇██████████
epoch/auc,▁▅▆▅▇▇█▇▇███████▇█████████████████
epoch/auc_1,▁▅▆▅▇▇█▇▇███████▇█████████████████
+9,...


...

[I 2026-03-20 08:22:46,294] Trial 30 finished with value: 0.0 and parameters: {'activation': 'leaky_relu', 'regularizer_value_2': 6.882307824100193e-06, 'N_layers': 17, 'learning_rate': 0.0006331128316787533, 'optimizer': 'adamw', 'N_1st_layer': 226, 'Dropout': 'n', 'N_1_layer': 141, 'Dropout_value_L2': 0.1298039257068876, 'N_2_layer': 164, 'Dropout_value_L3': 0.12377792767450743, 'N_3_layer': 149, 'Dropout_value_L4': 0.13086389220586656, 'N_4_layer': 180, 'Dropout_value_L5': 0.1407853081691064, 'N_5_layer': 228, 'Dropout_value_L6': 0.14916893211131277, 'N_6_layer': 203, 'Dropout_value_L7': 0.10692601817352457, 'N_7_layer': 185, 'Dropout_value_L8': 0.10760462793516538, 'N_8_layer': 158, 'Dropout_value_L9': 0.1062732120661408, 'N_9_layer': 146, 'Dropout_value_L10': 0.12843709936288872, 'N_10_layer': 180, 'Dropout_value_L11': 0.1496367809388913, 'N_11_layer': 212, 'Dropout_value_L12': 0.1332070350186622, 'N_12_layer': 138, 'Dropout_value_L13': 0.10998661492911979, 'N_13_layer': 136, 'Dro

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 12)]                 0         []                            
                                                                                                  
 dense (Dense)               (None, 204)                  2652      ['input_1[0][0]']             
                                                                                                  
 activation (Activation)     (None, 204)                  0         ['dense[0][0]']               
                                                                                                  
 batch_normalization (Batch  (None, 204)                  816       ['activation[0][0]']          
 Normalization)                                                                               

batch/accuracy,▂▁▂▂▃▄▅▃▄▅▄▅▅▅▁▅▄▆▆▆█▇▆▆▆▅▅▆▅▆██▇▆▆▇▇▆▆▆
batch/auc,▁▃▃▃▃▃▃▃▃▃▂▃▃▃▃▆▄▄▄▄▄▄▄▄▄▃▃▄▄▄▃▄▅▄▄▅█▆▆▆
batch/auc_1,▃▄▃▅▄▄█▁▃▃▃▃▄▃▄▄▅▅▄▄▅▇▅▇▁▃▄▄▄▄▄▄▄▅▅▅█▆▇▇
batch/batch_step,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇▇███
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,▆▆▆▆▆█▆▅▅▅▅▅▄█▄▃▃▃▃▃▁▂▃▃▃▃▃▃▃▃▂▂▂▂▂▁▂▁▁▂
batch/precision,▆▅▄▆▅▅█▅▅▅▅▁▆▆▅▄▅▅▆▅▇▅▅▂▄▅▅▅▅▇▆▆▅▅▆▇▆▆▆▆
epoch/accuracy,▁▅▆▇███▇
epoch/auc,▁▁▂▄▃▂▆█
epoch/auc_1,▂▁▂▄▃▂▅█
+9,...


[I 2026-03-20 08:23:29,727] Trial 31 finished with value: 0.25 and parameters: {'activation': 'relu6', 'regularizer_value_2': 7.5555797738899e-07, 'N_layers': 19, 'learning_rate': 0.0008875198749709292, 'optimizer': 'adam', 'N_1st_layer': 204, 'Dropout': 'y', 'N_1_layer': 222, 'Dropout_value_L2': 0.12082661650219075, 'N_2_layer': 201, 'Dropout_value_L3': 0.13541306402056852, 'N_3_layer': 165, 'Dropout_value_L4': 0.11025941339921728, 'N_4_layer': 202, 'Dropout_value_L5': 0.13249161593551412, 'N_5_layer': 241, 'Dropout_value_L6': 0.11493593998187408, 'N_6_layer': 181, 'Dropout_value_L7': 0.1391551474659517, 'N_7_layer': 151, 'Dropout_value_L8': 0.12214668171373465, 'N_8_layer': 202, 'Dropout_value_L9': 0.10083841941810988, 'N_9_layer': 223, 'Dropout_value_L10': 0.10214045059227451, 'N_10_layer': 164, 'Dropout_value_L11': 0.12679858360490984, 'N_11_layer': 192, 'Dropout_value_L12': 0.14252569195370632, 'N_12_layer': 158, 'Dropout_value_L13': 0.12649189807415018, 'N_13_layer': 167, 'Dropou

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 12)]                 0         []                            
                                                                                                  
 dense (Dense)               (None, 162)                  2106      ['input_1[0][0]']             
                                                                                                  
 activation (Activation)     (None, 162)                  0         ['dense[0][0]']               
                                                                                                  
 batch_normalization (Batch  (None, 162)                  648       ['activation[0][0]']          
 Normalization)                                                                               

In [ ]:
activation = "leaky_relu"
units1 = 177
units_per_1ayer = [197,187,195,187,131,177,128,150,173,160,197,169,155,204,188,204,160,175]
dropout = [0.11837708483323324,0.13737877910840118,0.1079956248790474,0.12698261124271906,0.12794531424007843,
           0.1296820415321237,0.133612397529674,0.12228020647719044,0.12220554795841516,0.12719421422552665,
           0.1442505597555257,0.10793263719860888,0.1320798929391686,0.10965275854307108,0.12948876012414212,
           0.10954741712920828,0.10762257284228328,0.1000578011775019]

r2 = 0.00000091842508974001
eta = 0.0008179841002734692

tf.keras.backend.clear_session()
inputs = layers.Input(shape = (X_train.shape[1],))
x = layers.Dense(units1, input_shape = (X_train.shape[1],))(inputs)
x = layers.Activation(activation)(x)
x = layers.BatchNormalization()(x)

for i in range(len(units_per_1ayer)):
    n = units_per_1ayer[i]
    # dropout_rate = 0
    dropout_rate = dropout[i]
    x = residual_block(x, n, activation, "t", dropout_rate, "L2", r2)            
            
x = layers.Dropout(0.4)(x)  
outputs = layers.Dense(1, activation = "sigmoid")(x)
model = models.Model(inputs, outputs)

model.compile(optimizer = tf.keras.optimizers.RMSprop(learning_rate = eta),
              loss = "binary_crossentropy",
              metrics = ["accuracy",
                         tf.keras.metrics.Precision(),
                         tf.keras.metrics.AUC(curve = "ROC"),
                         tf.keras.metrics.AUC(curve = "PR")])

wandb.init(project = "Residual-SnB(Balanced)-Trials-1.0",
           name = "Trial_1",
           reinit = True,
           config = {
               "Units_1": units1,
               "Units_per_layer": units_per_1ayer,
               "Droput": "yes",
               "Dropout_per_layer": dropout,
               "activation": activation,
               "n_layers": len(units_per_1ayer),
               "regularizer": "R2",
               "r_value2": r2,
               "learning_rate": eta,
               "optimizer": "Adam"})

print(model.summary())

early_stopping = EarlyStopping(monitor = "val_accuracy", patience = 7, restore_best_weights = True)
lr_reduction = ReduceLROnPlateau(monitor = "val_loss", factor = 0.1, patience = 5)

history = model.fit(X_train, y_train,
                    validation_data = (X_test, y_test),
                    batch_size = 32,
                    epochs = 200,
                    verbose = 1, 
                    callbacks = [WandbMetricsLogger(log_freq = 5), early_stopping, lr_reduction]
                   )

val_loss = min(history.history["val_loss"])
val_accuracy = max(history.history["val_accuracy"])
        
train_loss = min(history.history["loss"])
train_accuracy = max(history.history["accuracy"])

tf.keras.backend.clear_session()
gc.collect()
wandb.finish()

In [ ]:
model.save("ResNet-SnB-Higgs_B-Classificator-Balanced3.keras")

In [ ]:
model_ev = keras.models.load_model("ResNet-SnB-Higgs_B-Classificator-Balanced3.keras")

loss, accuracy, precision, auc, auc_1 = model_ev.evaluate(X_val, y_val, verbose = 1)

print(f"Test Loss: {loss:}")
print(f"Test Accuracy: {accuracy:}")

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = model_ev.predict(X_val)
y_pred_classes = (y_pred > 0.5).astype("int32")

cm = confusion_matrix(y_pred_classes, y_val)
sns.heatmap(cm, annot = True, fmt = "d", cmap = "rocket")

plt.xlabel("Prediction")
plt.ylabel("Real Value")
plt.show()